[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/spoken_dialogue_systems/spoken_dialogue_systems_lab.ipynb)

# Spoken Dialogue Systems — Lab

**Whisper ASR · Qwen conversational agent · MMS-TTS · Silero VAD**

- **Exercise 1 — The Pipeline: ASR:** speech recognition, background noise, Word Error Rate.
- **Exercise 2 — The Dialogue Components:** a multi-turn Cascade Dialogue System with conversational history.
- **Exercise 3 — VAD: Speech Segments:** threshold-based speech detection on recorded audio.

## Speech signals and pretrained components

- **Speech input:** mono waveform, **16 kHz**.
- **Pretrained models:** speech recognition, conversational agent, speech synthesis and VAD.
- **Outputs:** transcript, generated reply, synthesized speech, speech segments.

In [ ]:
# Compatible with the current Colab Hugging Face stack (Gradio and Diffusers).
# Transformers 5 supports Hugging Face Hub 1.x and the Qwen / MMS-TTS models.
%pip -q install --upgrade "transformers==5.22.2" "huggingface-hub>=1.32,<2" "accelerate>=1.1" "faster-whisper>=1.1,<2" "jiwer>=3,<5" "soundfile>=0.12" "sentencepiece>=0.2" "silero-vad>=5,<7" "ipywidgets>=8,<9"
!which ffmpeg >/dev/null || (apt-get -qq update && apt-get install -y -qq ffmpeg)

# Confirm the versions actually available in this runtime.
from importlib.metadata import version
print("transformers:", version("transformers"))
print("huggingface-hub:", version("huggingface-hub"))


In [ ]:
import base64
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display
from jiwer import wer, Compose, RemoveMultipleSpaces, RemovePunctuation, Strip, ToLowerCase

SR = 16_000  # use 16 kHz mono for both Whisper and Silero
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

normalize = Compose([ToLowerCase(), RemovePunctuation(), RemoveMultipleSpaces(), Strip()])
def word_error_rate(reference, prediction):
    return 100 * wer(normalize(reference), normalize(prediction))

### Speech input

A microphone recording contains **words, pauses and background noise**. `record_audio()` provides a 16 kHz waveform used by ASR and VAD.

In [ ]:
# Provided helper. Students do not need to edit this function.
def record_audio(filename="recording.wav"):
    """Record in Google Colab; fall back to file upload when recording is blocked."""
    from google.colab import output, files

    js = r"""
    new Promise(async (resolve, reject) => {
      try {
        const stream = await navigator.mediaDevices.getUserMedia({audio: true});
        const formats = ['audio/webm;codecs=opus', 'audio/webm', 'audio/mp4'];
        const mime = formats.find(x => MediaRecorder.isTypeSupported(x));
        const recorder = mime ? new MediaRecorder(stream, {mimeType: mime})
                              : new MediaRecorder(stream);
        const chunks = [];
        const div = document.createElement('div');
        const start = document.createElement('button');
        const stop = document.createElement('button');
        const status = document.createElement('span');
        start.textContent = '● Start recording';
        stop.textContent = '■ Stop recording';
        stop.disabled = true;
        status.textContent = '  Ready';
        div.append(start, stop, status);
        document.body.appendChild(div);
        recorder.ondataavailable = e => {if (e.data.size > 0) chunks.push(e.data);};
        recorder.onstop = () => {
          stream.getTracks().forEach(track => track.stop());
          status.textContent = '  Saving...';
          const blob = new Blob(chunks, {type: recorder.mimeType});
          const reader = new FileReader();
          reader.onloadend = () => {div.remove(); resolve(reader.result);};
          reader.onerror = reject;
          reader.readAsDataURL(blob);
        };
        start.onclick = () => {recorder.start(); start.disabled = true;
                               stop.disabled = false; status.textContent = '  Recording...';};
        stop.onclick = () => {stop.disabled = true; recorder.stop();};
      } catch (err) {reject(err.toString());}
    })
    """
    try:
        data_url = output.eval_js(js)  # waits until the student presses Stop
        prefix, encoded = data_url.split(",", 1)
        suffix = ".mp4" if "mp4" in prefix else ".webm"
        original = Path(filename).with_suffix(suffix)
        original.write_bytes(base64.b64decode(encoded))
    except Exception as error:
        print("Microphone unavailable:", error)
        print("Upload a short WAV, MP3, M4A, or WebM recording instead.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No recording provided")
        original = Path(next(iter(uploaded)))

    wav_file = Path(filename)
    if original.resolve() == wav_file.resolve():
        wav_file = wav_file.with_name(wav_file.stem + "_converted.wav")
    subprocess.run(
        ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(original),
         "-ac", "1", "-ar", str(SR), str(wav_file)], check=True
    )
    audio, sample_rate = sf.read(wav_file, dtype="float32")
    assert sample_rate == SR and audio.ndim == 1
    if len(audio) < SR // 2:
        print("Recording is very short. Consider recording it again.")
    return audio

# Exercise 1 — The Pipeline: ASR

**ASR:** speech signal → sequence of words. Whisper is an **encoder–decoder** model.

**Reference transcripts** (record your own voice):
1. “I want to book a train to Milan.”
2. “I need a ticket from Verona to Trento.”
3. “What time does the next train leave?”

**Compare:** reference transcript vs. Whisper transcript.

In [ ]:
SENTENCES = [
    "I want to book a train to Milan.",
    "I need a ticket from Verona to Trento.",
    "What time does the next train leave?",
]

recordings = []
for i, sentence in enumerate(SENTENCES, 1):
    display(Markdown(f"**Sentence {i}:** {sentence}"))
    audio = record_audio(f"sentence_{i}.wav")
    recordings.append(audio)
    display(Audio(audio, rate=SR))

### ASR: Word Error Rate (WER)

$$
\mathrm{WER}\;(\%) = 100 \times \frac{S + D + I}{N}
$$

- **S:** Substitutions
- **D:** Deletions
- **I:** Insertions
- **N:** Number of words in the reference transcript
- **WER = 0%:** No word errors after normalization.

In [ ]:
from faster_whisper import WhisperModel

# Download and load the tiny English ASR model once.
asr_device = DEVICE
asr = WhisperModel("tiny.en", device=asr_device,
                   compute_type="float16" if asr_device == "cuda" else "int8")

def transcribe(audio):
    """Return a string from a 16 kHz mono NumPy waveform."""
    global asr, asr_device
    samples = np.asarray(audio, dtype=np.float32)
    try:
        segments, _ = asr.transcribe(samples, language="en", beam_size=1,
                                     vad_filter=False, condition_on_previous_text=False)
        return " ".join(s.text.strip() for s in segments).strip()
    except Exception as e:
        if asr_device != "cuda":
            raise
        print("Whisper CUDA unavailable; using CPU:", str(e)[:100])
        asr = WhisperModel("tiny.en", device="cpu", compute_type="int8")
        asr_device = "cpu"
        segments, _ = asr.transcribe(samples, language="en", beam_size=1,
                                     vad_filter=False, condition_on_previous_text=False)
        return " ".join(s.text.strip() for s in segments).strip()

In [ ]:
# Transcribe each clean recording and compare with its reference.
clean_results = []
for i, (reference, audio) in enumerate(zip(SENTENCES, recordings), 1):
    prediction = transcribe(audio)
    clean_results.append({"Sentence": i, "Reference": reference,
                          "Whisper transcription": prediction,
                          "WER (%)": round(word_error_rate(reference, prediction), 1)})
display(pd.DataFrame(clean_results))

### ASR: Recognition Errors with Background Noise

**Gaussian noise** is added to each recording (function provided).

$$
\mathrm{SNR}_{\mathrm{dB}} = 10 \log_{10}\left(\frac{P_{\mathrm{speech}}}{P_{\mathrm{noise}}}\right)
$$

- **20, 10, 0 dB:** Decreasing signal-to-noise ratio.
- **Compare:** Clean/noisy audio, recognized words, and WER.
- **Note:** WER may vary non-monotonically across recordings.

In [ ]:
# Provided: no TODOs here.
def add_gaussian_noise(audio, snr_db, seed=42):
    audio = np.asarray(audio, dtype=np.float32)
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(audio.shape).astype(np.float32)
    speech_power = np.mean(audio ** 2)
    noise_power = np.mean(noise ** 2)
    if speech_power == 0 or noise_power == 0:
        return audio.copy()
    target_noise_power = speech_power / (10 ** (snr_db / 10))
    return audio + noise * np.sqrt(target_noise_power / noise_power)

In [ ]:
noise_results = []
noisy_recordings = {}
for i, (reference, audio) in enumerate(zip(SENTENCES, recordings), 1):
    for snr_db in [20, 10, 0]:
        noisy = add_gaussian_noise(audio, snr_db)
        noisy_recordings[(i, snr_db)] = noisy
        transcript = transcribe(noisy)
        noise_results.append({
            "Sentence": i,
            "SNR (dB)": snr_db,
            "Whisper transcript": transcript,
            "WER (%)": round(word_error_rate(reference, transcript), 1),
        })
display(pd.DataFrame(noise_results))

# Compare what you hear with what Whisper recognized.
LISTEN_SENTENCE = 1       # 1, 2, or 3
LISTEN_SNR = 0            # 20, 10, or 0 dB
print("Clean speech:")
display(Audio(recordings[LISTEN_SENTENCE - 1], rate=SR))
print(f"Speech with Gaussian noise ({LISTEN_SNR} dB):")
display(Audio(noisy_recordings[(LISTEN_SENTENCE, LISTEN_SNR)], rate=SR))

**Discuss:** Which substitutions, deletions or insertions appear? Do errors in *Milan* or *Trento* change the request?

# Exercise 2 — The Dialogue Components

**Cascade Dialogue System:**

**User voice request → ASR (Whisper) → Words → LLM (Qwen conversational agent) → Words → TTS → Voice reply**

- **ASR:** turns the speech signal into words.
- **Conversational agent:** uses an LLM for NLU, DM and NLG.
- **TTS:** turns the generated reply into a speech waveform.
- **Conversational history:** the user transcript and agent reply are added after every turn. The next LLM call receives the earlier turns.
- **System prompt:** defines the conversational agent's behaviour.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, VitsModel

QWEN_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TTS_NAME = "facebook/mms-tts-eng"

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_NAME)
qwen = AutoModelForCausalLM.from_pretrained(
    QWEN_NAME, dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE).eval()

tts_tokenizer = AutoTokenizer.from_pretrained(TTS_NAME)
tts = VitsModel.from_pretrained(TTS_NAME).to(DEVICE).eval()
print("Loaded:", QWEN_NAME, "and", TTS_NAME)

In [ ]:
# Provided pretrained-model helpers.
def qwen_reply(user_text, system_prompt, history=None):
    """Generate an LLM reply conditioned on the system prompt and previous turns."""
    if not user_text.strip():
        return "I could not hear you. Could you repeat your question?"
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history if history is not None else [])
    messages.append({"role": "user", "content": user_text})
    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        result_ids = qwen.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    reply_ids = result_ids[0, inputs["input_ids"].shape[-1]:]
    return qwen_tokenizer.decode(reply_ids, skip_special_tokens=True).strip()


def synthesize(text):
    """Return (speech waveform, native sample rate) from the TTS model."""
    inputs = tts_tokenizer(text[:250], return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        speech = tts(**inputs).waveform[0].float().cpu().numpy()
    return speech, tts.config.sampling_rate

### Build a multi-turn conversational agent

- Connect **ASR → conversational agent → TTS** in `spoken_agent()`.
- The **loop** collects voice requests, plays the voice replies and adds each user/agent turn to the **conversational history**.
- Observe the **ASR transcript**, **generated reply** and **synthesized speech** at each turn.
- Change the **system prompt** and compare the conversational responses using the **same recorded requests**. (The history starts empty for each comparison.)

**Question:** Does the conversational agent use information from earlier turns?

In [ ]:
# Exercise 2 — connect the three components; the history structure is provided.
def spoken_agent(audio, system_prompt, history):
    # TODO: convert the speech signal into a transcript with Whisper.
    # transcript = ...

    # TODO: generate the reply, passing the previous conversational history.
    # reply = ...

    # TODO: synthesize the reply into a waveform and its sampling rate.
    # speech, speech_sr = ...

    # The next turn must include both sides of this exchange.
    # updated_history = history + [
    #     {"role": "user", "content": transcript},
    #     {"role": "assistant", "content": reply},
    # ]
    # return {"transcript": transcript, "reply": reply,
    #         "speech": speech, "speech_sr": speech_sr,
    #         "history": updated_history}
    raise NotImplementedError("Connect ASR, conversational agent, TTS and history")

In [ ]:
# One system prompt for the entire conversation. Change it and compare responses.
SYSTEM_PROMPT = (
    "You are a helpful train-travel conversational agent. "
    "Remember the details mentioned in previous turns. "
    "Reply in short, natural English. Do not invent live train schedules."
)

N_TURNS = 3
RECORD_NEW_AUDIO = True  # False: reuse recorded_turns to compare another system prompt

if RECORD_NEW_AUDIO:
    recorded_turns = []
else:
    assert "recorded_turns" in globals() and len(recorded_turns) == N_TURNS, (
        "A conversation with N_TURNS recorded inputs is needed."
    )

history = []  # a new conversation starts with an empty history
for turn in range(N_TURNS):
    print(f"\n--- Turn {turn + 1} ---")
    if RECORD_NEW_AUDIO:
        audio = record_audio(f"dialogue_turn_{turn + 1}.wav")
        recorded_turns.append(audio)
    else:
        audio = recorded_turns[turn]

    display(Audio(audio, rate=SR))
    result = spoken_agent(audio, SYSTEM_PROMPT, history)
    history = result["history"]  # previous user/agent turns are retained

    print("ASR transcript:", result["transcript"])
    print("Conversational agent reply:", result["reply"])
    print("TTS voice reply:")
    display(Audio(result["speech"], rate=result["speech_sr"]))

print("\nNumber of exchanges stored in the history:", len(history) // 2)

**Discuss:** How does **conversational history** affect the second and third replies? Which output changes when the **system prompt** changes? How might an ASR recognition error propagate to the agent's reply?

# Exercise 3 — VAD: Speech Segments

- **Voice Activity Detection (VAD):** identifies **when someone is speaking**, not the words.
- **Speech segments:** start and end of detected speech in the audio signal.
- **Threshold:** controls how confidently Silero identifies speech.

**Exercise:** record a short audio sample with a few pauses. Compare the detected speech regions on the waveform with the audio obtained by concatenating those regions.

In [ ]:
# Your own voice: any words, with a few pauses between phrases.
vad_audio = record_audio("vad_recording.wav")
print(f"Recording duration: {len(vad_audio) / SR:.1f} s")
print("Original recording:")
display(Audio(vad_audio, rate=SR))

In [ ]:
from silero_vad import load_silero_vad, get_speech_timestamps

vad_model = load_silero_vad()


def show_speech_segments(audio, threshold=0.5):
    """Detect speech, show the selected regions, and play concatenated speech."""
    audio = np.asarray(audio, dtype=np.float32)
    vad_model.reset_states()
    speech_segments = get_speech_timestamps(
        torch.from_numpy(audio),
        vad_model,
        sampling_rate=SR,
        threshold=threshold,
    )

    # Keep only the parts of the recording classified as speech.
    clips = [audio[s["start"]:s["end"]] for s in speech_segments]
    speech_only = np.concatenate(clips) if clips else np.array([], dtype=np.float32)

    fig, axes = plt.subplots(2, 1, figsize=(11, 4.5))
    axes[0].plot(np.arange(len(audio)) / SR, audio, color="0.35", lw=0.6)
    for i, segment in enumerate(speech_segments):
        axes[0].axvspan(
            segment["start"] / SR, segment["end"] / SR,
            color="tab:green", alpha=0.30,
            label="Detected speech" if i == 0 else None,
        )
    axes[0].set(title="Original recording — detected speech in green",
                xlabel="Time (s)", ylabel="Amplitude")
    axes[0].set_xlim(0, len(audio) / SR)
    if speech_segments:
        axes[0].legend(loc="upper right")

    if len(speech_only):
        axes[1].plot(np.arange(len(speech_only)) / SR, speech_only,
                     color="tab:green", lw=0.6)
    axes[1].set(title="Concatenated detected speech",
                xlabel="Time (s)", ylabel="Amplitude")
    axes[1].set_xlim(0, max(len(speech_only) / SR, 0.1))
    fig.tight_layout()
    plt.show()

    print("Detected speech intervals (s):", [
        (round(s["start"] / SR, 2), round(s["end"] / SR, 2))
        for s in speech_segments
    ])
    if len(speech_only):
        print("Audio containing only the detected speech segments:")
        display(Audio(speech_only, rate=SR))
    else:
        print("No speech detected at this threshold.")
    return speech_segments

### VAD: Speech Segments

The **green areas** are classified as speech by Silero. The second waveform and playback contain only those regions, joined together.

Compare the output with the original recording. Change `THRESHOLD` to **0.3** or **0.7** and observe which parts of the speech signal are retained.

In [ ]:
THRESHOLD = 0.5  # Try 0.3 and 0.7
speech_segments = show_speech_segments(vad_audio, threshold=THRESHOLD)

**Discuss:** Does Silero detect every spoken phrase? What changes when the threshold increases? Can it identify **speech segments** without recognizing any words?

**Barge-in:** in a spoken dialogue system, VAD can indicate when a user starts speaking while the system is responding. This exercise detects speech in a recorded signal, not live interruption.